# 🛡️ Fine-Tuning BanglaBERT for Toxic & Harassment Comment Detection

Train BUET NLP's state-of-the-art `csebuetnlp/banglabert` model on 50,281 real Bengali social media comments.

- **Dataset:** 40,224 train / 5,028 val / 5,029 test
- **Task:** Binary Toxicity & Harassment Classification (`0: Non-Toxic`, `1: Toxic / Hate Speech`)
- **Target Demographic:** Female, Male, Individual, Group cyberbullying
- **Author:** Shaiera Sultana Oishe  
- **Recommended Runtime:** GPU (Google Colab T4 / V100)

### Step 1: Install Required Libraries

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas torch huggingface_hub

### Step 2: Verify GPU Acceleration
Ensure you have selected **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device}")
if device == "cuda":
    print(f"⚡ GPU Model: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: No GPU detected! Go to Runtime > Change runtime type and select T4 GPU.")

### Step 3: Unpack Dataset
Upload `downloaded_dataset.zip` to the left files panel in Colab, or upload the `downloaded_dataset` folder.

In [ ]:
import os
import zipfile
import pandas as pd

# Auto-unzip if zip file is uploaded
if os.path.exists("downloaded_dataset.zip") and not os.path.exists("downloaded_dataset"):
    print("Unzipping downloaded_dataset.zip...")
    with zipfile.ZipFile("downloaded_dataset.zip", 'r') as zip_ref:
        zip_ref.extractall(".")

train_df = pd.read_csv("downloaded_dataset/train.csv").dropna(subset=['sentence', 'hate speech'])
val_df = pd.read_csv("downloaded_dataset/val.csv").dropna(subset=['sentence', 'hate speech'])
test_df = pd.read_csv("downloaded_dataset/test.csv").dropna(subset=['sentence', 'hate speech'])

print(f"✅ Train samples : {len(train_df):,}")
print(f"✅ Val samples   : {len(val_df):,}")
print(f"✅ Test samples  : {len(test_df):,}")
print("\nClass balance in Train:")
print(train_df['hate speech'].value_counts())

### Step 4: Tokenization with BanglaBERT (`csebuetnlp/banglabert`)

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "csebuetnlp/banglabert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_ds = Dataset.from_pandas(train_df[['sentence', 'hate speech']].rename(columns={'hate speech': 'label'}))
val_ds = Dataset.from_pandas(val_df[['sentence', 'hate speech']].rename(columns={'hate speech': 'label'}))
test_ds = Dataset.from_pandas(test_df[['sentence', 'hate speech']].rename(columns={'hate speech': 'label'}))

print("Tokenizing datasets...")
train_tokenized = train_ds.map(tokenize_fn, batched=True, batch_size=2000)
val_tokenized = val_ds.map(tokenize_fn, batched=True, batch_size=2000)
test_tokenized = test_ds.map(tokenize_fn, batched=True, batch_size=2000)
print("✅ Tokenization complete!")

### Step 5: Initialize Model Architecture

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Non-Toxic", 1: "Toxic / Harassment"},
    label2id={"Non-Toxic": 0, "Toxic / Harassment": 1},
    classifier_dropout=0.2
)
model.to(device)
print(f"✅ Initialized {MODEL_NAME} on {device}")

### Step 6: Define Evaluation Metrics (Macro-F1, Precision, Recall, Accuracy)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": round(acc, 4),
        "macro_f1": round(f1, 4),
        "macro_precision": round(precision, 4),
        "macro_recall": round(recall, 4)
    }

### Step 7: Train the Model

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./banglabert_toxic_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=300,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("🚀 Starting Training (Approx. 15-25 minutes on T4 GPU)...\n")
trainer.train()

### Step 8: Evaluate on Holdout Test Set (Unseen 5,029 comments)

In [ ]:
print("=== Final Evaluation on Holdout Test Set ===")
test_results = trainer.evaluate(test_tokenized)
for k, v in test_results.items():
    print(f"{k}: {v}")

### Step 9: Save and Push to Hugging Face Hub
Publishing your model to Hugging Face gives you a permanent model card that anyone can load via `from_pretrained("your-name/banglabert-toxic-detector")`!

In [ ]:
# Save model locally in Colab
model.save_pretrained("./best_banglabert_model")
tokenizer.save_pretrained("./best_banglabert_model")
print("✅ Saved locally in ./best_banglabert_model")

# Optional: Push directly to your Hugging Face account
# from huggingface_hub import login
# login()  # Paste your HF write token
# HF_REPO_NAME = "your-username/banglabert-toxic-harassment-detector"
# model.push_to_hub(HF_REPO_NAME)
# tokenizer.push_to_hub(HF_REPO_NAME)
# print(f"🎉 Model published to https://huggingface.co/{HF_REPO_NAME}")